In [ ]:
## Setup — packages & environment

import sys, subprocess
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'networkx', 'openpyxl', 'reportlab']
for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except:
        install(pkg)

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
import os, datetime as dt

RSEED = 2023
np.random.seed(RSEED)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel installed')
except:
    print('ipykernel not installed')

In [ ]:
## 1. Network Data and Construction

np.random.seed(RSEED)
n_nodes = 30

# Create edge list (forum interactions)
edges = []
for i in range(n_nodes):
    for j in range(i+1, n_nodes):
        if np.random.rand() < 0.2:  # 20% edge density
            weight = np.random.randint(1, 5)
            edges.append((f'student_{i}', f'student_{j}', weight))

# Create network
G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# Add isolated nodes
for i in range(n_nodes):
    if f'student_{i}' not in G:
        G.add_node(f'student_{i}')

print('Network Statistics:')
print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print(f'Density: {nx.density(G):.3f}')
print(f'Average Clustering: {nx.average_clustering(G):.3f}')

In [ ]:
## 2. Centrality Metrics Computation

# Compute centrality measures
degree_cent = nx.degree_centrality(G)
between_cent = nx.betweenness_centrality(G, weight='weight')
close_cent = nx.closeness_centrality(G, distance='weight')
eigen_cent = nx.eigenvector_centrality(G, max_iter=100, weight='weight')

# Compile results
centrality_data = pd.DataFrame({
    'node': list(G.nodes()),
    'degree': [degree_cent[n] for n in G.nodes()],
    'betweenness': [between_cent[n] for n in G.nodes()],
    'closeness': [close_cent[n] for n in G.nodes()],
    'eigenvector': [eigen_cent[n] for n in G.nodes()]
})

print('Top 10 Nodes by Degree Centrality:')
print(centrality_data.nlargest(10, 'degree')[['node', 'degree']])

centrality_data.to_csv('network_centralities.csv', index=False)
print('\nSaved centrality data')

In [ ]:
## 3. Visualizations

os.makedirs('figures', exist_ok=True)

# 1. Network visualization with centrality coloring
fig, ax = plt.subplots(figsize=(12, 10))
pos = nx.spring_layout(G, k=0.5, iterations=50, seed=RSEED)

# Node colors based on degree centrality
node_colors = [degree_cent[n] for n in G.nodes()]
node_sizes = [500 * degree_cent[n] + 100 for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                       cmap='YlOrRd', ax=ax, alpha=0.8)
nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax, width=0.5)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', ax=ax)

sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(vmin=min(node_colors), vmax=max(node_colors)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Degree Centrality', fontsize=11)

ax.set_title('Course Interaction Network (Nodes Colored by Degree Centrality)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('figures/01_network_visualization.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_network_visualization.png')

# 2. Centrality comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
top_n = 10
top_nodes = centrality_data.nlargest(top_n, 'degree')['node']

for idx, (ax, metric) in enumerate(zip(axes.flat, ['degree', 'betweenness', 'closeness', 'eigenvector'])):
    data = centrality_data[centrality_data['node'].isin(top_nodes)].sort_values(metric, ascending=True)
    colors = sns.color_palette('viridis', len(data))
    ax.barh(data['node'], data[metric], color=colors, edgecolor='black', alpha=0.8)
    ax.set_xlabel(metric.capitalize(), fontsize=11)
    ax.set_title(f'Top {top_n} by {metric.capitalize()}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('figures/02_centrality_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_centrality_comparison.png')

# 3. Degree distribution
degrees = [G.degree(n) for n in G.nodes()]
fig, ax = plt.subplots(figsize=(9, 6))
ax.hist(degrees, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(degrees), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(degrees):.2f}')
ax.set_xlabel('Degree', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Degree Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('figures/03_degree_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 03_degree_distribution.png')

In [ ]:
## 4. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch15_SocialNetworkAnalysis_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 15: Social Network Analysis</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Social network analysis (SNA) examines relationships and interaction patterns among course participants. '
    'Centrality metrics identify influential students and knowledge brokers in the learning community.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Network Properties</b>', styles['Heading2']))
props = f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Density: {nx.density(G):.3f}'
story.append(Paragraph(props, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/01_network_visualization.png'):
        story.append(Image('figures/01_network_visualization.png', width=520, height=420))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Course interaction network.', styleN))
        story.append(Spacer(1, 12))
except: pass

try:
    if os.path.exists('figures/03_degree_distribution.png'):
        story.append(Image('figures/03_degree_distribution.png', width=480, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 2: Degree distribution.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>4. Interpretation</b>', styles['Heading2']))
interp = (
    'Degree centrality identifies highly connected students. '
    'Betweenness identifies brokers who bridge student groups. '
    'Closeness measures quick information spread from a node.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')